# Project Bài 5 · Tự viết bộ phân lớp logistic dự báo đơn trễ

**Học máy cơ bản · Bài 5 · PGS.TS Nguyễn Văn Hậu**

**Tình huống.** Bộ phận chăm sóc khách hàng muốn gọi trước cho một số ít đơn Olist có nguy cơ trễ, ngay lúc duyệt đơn. Bạn tự viết toàn bộ thuật toán học của hồi quy logistic (sigmoid, BCE, gradient, gradient descent có L2), chọn ngưỡng cảnh báo đúng quy trình và viết báo cáo một trang có giới hạn.

**Dữ liệu.** `data/olist_model_table.csv`: cùng bảng, cùng sáu đặc trưng và cùng ba tập với Bài 4 và slide Bài 5 (train 29.031 · validation 10.389 · test 40.266 đơn). Cột `late` là nhãn theo ngày lịch.

| Mốc | Bước | Ô tự kiểm |
|---|---|---|
| M1 | 1–2: dữ liệu; sigmoid, BCE, gradient | số đơn và số đơn trễ; giá trị sigmoid; sai phân hữu hạn |
| M2 | 3: gradient descent có L2 | bốn đơn A–D; J giảm đơn điệu; (khá) khớp scikit-learn dưới 10⁻⁶ |
| M3 | 4–5: chọn τ trên validation, chấm test một lần | ngưỡng được chọn; `cham_test` gọi đúng một lần |

**Ba mức.** *Đạt*: bước 1–5 qua ô tự kiểm (đặt `LAM_MUC_KHA = False`). *Khá*: GD khớp nghiệm scikit-learn giải chặt với sai lệch hệ số dưới 10⁻⁶ và giải thích vai trò của C và η. *Nâng cao*: phần cuối (hiệu chuẩn, τ* = 1/6, thiên lệch do quy tắc biết nhãn).

**Luật chơi.** Không dùng `LogisticRegression` để học hệ số của bài nộp; chỉ dùng để đối chiếu ở ô có ghi rõ. **Chọn τ chỉ trên validation.** **Test chỉ chấm một lần** bằng `cham_test`. Chi phí FP + 5·FN là giả định lớp học.

In [ ]:
# Thiết lập: chạy ô này trước
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
assert (ROOT / 'data' / 'olist_model_table.csv').exists(), 'Mở notebook từ thư mục bài (có thư mục data/).'
FEAT = ['distance_km', 'promised_days', 'price', 'freight_value', 'n_items', 'weight_g']
DATES = ['order_approved_at', 'order_delivered_customer_date', 'order_estimated_delivery_date']

def kiem(ten, dieu_kien, goi_y=''):
    # Ô tự kiểm: in ✅ khi đúng; báo lỗi kèm gợi ý khi sai.
    if not dieu_kien:
        raise AssertionError(f'❌ {ten}. Gợi ý: {goi_y}')
    print(f'✅ {ten}')
LAM_MUC_KHA = True

## Bước 1 · Dữ liệu, nhãn và ba tập (Chặng 1)

Nhãn: `late = 1` khi ngày giao (bỏ giờ) sau ngày hẹn (bỏ giờ). Đặc trưng chỉ gồm sáu cột biết lúc duyệt đơn.

In [ ]:
d = pd.read_csv(ROOT / 'data' / 'olist_model_table.csv', parse_dates=DATES)
tr, va, te = (d[d.split == k].reset_index(drop=True) for k in ['train', 'validation', 'test'])
# TODO: tạo lại nhãn trễ theo ngày lịch từ hai cột ngày (bỏ giờ bằng .dt.normalize())
nhan_tu_tinh = ...
raise NotImplementedError('TODO bước 1: tạo nhan_tu_tinh rồi xoá dòng raise này')
print({k: (len(a), int(a.late.sum())) for k, a in [('train', tr), ('validation', va), ('test', te)]})

In [ ]:
# TỰ KIỂM · bước 1
kiem('Số đơn ba tập', (len(tr), len(va), len(te)) == (29031, 10389, 40266), 'Lọc theo cột split.')
kiem('Số đơn trễ ba tập', (tr.late.sum(), va.late.sum(), te.late.sum()) == (941, 634, 3478), '')
kiem('Nhãn tự tạo khớp cột late', (nhan_tu_tinh.values == d.late.values).all(), 'So ngày, không so thời điểm có giờ.')
kiem('Không có cột kết quả trong đặc trưng', not any('delivered' in c or 'carrier' in c for c in FEAT), '')

### Tiền xử lý (cho sẵn): median và chuẩn hoá chỉ fit trên train

In [ ]:
median = tr[FEAT].median()
mu = tr[FEAT].fillna(median).mean()
sd = tr[FEAT].fillna(median).std(ddof=0)           # StandardScaler dùng độ lệch chuẩn tổng thể (ddof = 0)
chuan = lambda a: ((a[FEAT].fillna(median) - mu) / sd).to_numpy()
Xtr, Xva, Xte = chuan(tr), chuan(va), chuan(te)
ytr, yva, yte = tr.late.to_numpy(), va.late.to_numpy(), te.late.to_numpy()
print(Xtr.shape, Xva.shape, Xte.shape, '· trung bình cột train ≈ 0:', np.allclose(Xtr.mean(0), 0))

## Bước 2 · Sigmoid, BCE và gradient (Chặng 1–3)

- `sigmoid(z)`: không để `np.exp` tràn số với z rất âm (gợi ý: tách hai trường hợp z ≥ 0 và z < 0, hoặc dùng `np.exp(-np.abs(z))`).
- `bce(b, w, X, y, lam)`: J = trung bình ℓ + λ/2·‖w‖²; viết ℓ ổn định số qua `np.logaddexp(0, z) − y·z`.
- `gradient(b, w, X, y, lam)`: trả (g_b, g_w) với g_b = trung bình (p − y), g_w = Xᵀ(p − y)/n + λw. Không phạt b.

In [ ]:
def sigmoid(z):
    # TODO: σ(z) = 1/(1 + e^(−z)), ổn định số
    raise NotImplementedError('TODO bước 2: sigmoid')

def bce(b, w, X, y, lam=0.0):
    # TODO: trung bình BCE + lam/2·||w||²
    raise NotImplementedError('TODO bước 2: bce')

def gradient(b, w, X, y, lam=0.0):
    # TODO: trả (g_b, g_w)
    raise NotImplementedError('TODO bước 2: gradient')

In [ ]:
# TỰ KIỂM · bước 2
kiem('σ(0) = 0,5 và σ(−0,8) ≈ 0,310', abs(sigmoid(0) - 0.5) < 1e-12 and abs(sigmoid(-0.8) - 0.310025518872) < 1e-12, '')
kiem('sigmoid không tràn số', np.isfinite(sigmoid(np.array([-800.0, 800.0]))).all() and sigmoid(-800.0) >= 0, 'Dùng exp(−|z|).')
XA, yA = np.array([[-1.0], [0.0], [1.0], [2.0]]), np.array([0, 0, 1, 1])
kiem('Bốn đơn A–D: J(0; 0) = ln 2', abs(bce(0, np.zeros(1), XA, yA) - np.log(2)) < 1e-12, '')
gb, gw = gradient(0, np.zeros(1), XA, yA)
kiem('Bốn đơn A–D: g_b = 0, g_w = −0,5', abs(gb) < 1e-12 and abs(gw[0] + 0.5) < 1e-12, 'Trung bình (p − y) và (p − y)·x.')
rng = np.random.default_rng(0); b0, w0 = 0.3, rng.normal(size=6) * 0.1; lam0 = 1 / len(ytr)
gb, gw = gradient(b0, w0, Xtr, ytr, lam0); h = 1e-6
fd = np.array([(bce(b0, w0 + h * e, Xtr, ytr, lam0) - bce(b0, w0 - h * e, Xtr, ytr, lam0)) / (2 * h) for e in np.eye(6)])
kiem('Gradient khớp sai phân hữu hạn trên Olist', np.allclose(gw, fd, atol=1e-7), 'Kiểm lại dấu và phần λw.')

## Bước 3 · Gradient descent có L2 (Chặng 3 · thuật toán slide 38)

Viết `gd(X, y, eta, lam, T)` bắt đầu từ b = 0, w = 0; mỗi vòng cập nhật **đồng thời** b và w; trả `b, w, lich_su_J`.
Trên Olist dùng λ = 1/(n·C) với C = 1, η = 1, T = 4.000: đây đúng là hàm mục tiêu của `LogisticRegression(C=1)` chia cho nC.

In [ ]:
def gd(X, y, eta=1.0, lam=0.0, T=1000):
    # TODO: lặp T vòng; lưu J trước mỗi bước; cập nhật đồng thời
    raise NotImplementedError('TODO bước 3: gd')
LAM = 1 / (len(ytr) * 1.0)
b_gd, w_gd, J_gd = gd(Xtr, ytr, eta=1.0, lam=LAM, T=4000)
print('b =', round(b_gd, 6), '· w =', np.round(w_gd, 6))
plt.semilogy(J_gd - J_gd.min() + 1e-16); plt.xlabel('vòng'); plt.ylabel('J − min J'); plt.title('GD tự viết trên train'); plt.show()

In [ ]:
# TỰ KIỂM · bước 3
bA, wA, _ = gd(XA, yA, eta=1.0, lam=0.0, T=1)
kiem('A–D, một bước: (b; w) = (0; 0,5)', abs(bA) < 1e-12 and abs(wA[0] - 0.5) < 1e-12, 'Cập nhật đồng thời từ gradient cũ.')
kiem('A–D, J sau 1 bước = 0,4886', abs(bce(bA, wA, XA, yA) - 0.488640709110) < 1e-10, '')
kiem('J trên Olist giảm đơn điệu', np.all(np.diff(J_gd) <= 1e-12), 'η quá lớn? Thử η nhỏ hơn.')
if LAM_MUC_KHA:
    W_DICH, B_DICH = np.array([0.483545, -0.475858, 0.044845, 0.007322, -0.038935, 0.091672]), -3.486828   # LogisticRegression(C=1, solver='newton-cg', tol=1e-12)
    kiem('Mức khá · khớp nghiệm scikit-learn giải chặt (sai lệch < 1e-6)', np.max(np.abs(np.r_[b_gd - B_DICH, w_gd - W_DICH])) < 1e-6,
         'Không phạt b; λ = 1/(nC); đủ vòng lặp.')

### Đối chiếu với scikit-learn (mức khá, chỉ để kiểm, không dùng làm bài nộp)

In [ ]:
from sklearn.linear_model import LogisticRegression
sk = LogisticRegression(C=1.0, solver='newton-cg', tol=1e-12, max_iter=1000).fit(Xtr, ytr)
print('lệch lớn nhất:', np.max(np.abs(np.r_[b_gd - sk.intercept_[0], w_gd - sk.coef_[0]])))
print('tổng p trên train = tổng y?', round(sigmoid(b_gd + Xtr @ w_gd).sum(), 4), ytr.sum())

## Bước 4 · Chọn τ trên validation (Chặng 4)

Lưới khai báo trước: {0,05; 0,10; 0,15; 0,20; 0,30; 0,50}. Chi phí giả định c_FP·FP + c_FN·FN với (1; 5). Quy tắc hoà: ít cảnh báo hơn, rồi ngưỡng lớn hơn. Luôn so với mốc “không bao giờ cảnh báo”.

In [ ]:
LUOI = [0.05, 0.10, 0.15, 0.20, 0.30, 0.50]
p_va = sigmoid(b_gd + Xva @ w_gd)
def ma_tran(y, canh_bao):
    # TODO: trả (tn, fp, fn, tp), lớp dương = trễ
    raise NotImplementedError('TODO bước 4: ma_tran')

def chon_tau(p, y, c_fp=1, c_fn=5, luoi=LUOI):
    # TODO: quét lưới, tính chi phí, chọn theo (chi phí, số cảnh báo, −τ); trả (tau, bảng)
    raise NotImplementedError('TODO bước 4: chon_tau')
TAU, bang = chon_tau(p_va, yva)
khong_canh_bao = 5 * yva.sum()
print(bang.to_string(index=False)); print('τ đã chọn:', TAU, '· không cảnh báo:', khong_canh_bao)

In [ ]:
# TỰ KIỂM · bước 4
kiem('Chọn τ = 0.10'.replace('.', ','), TAU == 0.1, 'Kiểm quy tắc hoà và chiều so sánh p ≥ τ.')
kiem('Chi phí validation tại τ đã chọn = 3109', int(bang.loc[bang.tau == TAU, 'chi_phi'].iloc[0]) == 3109, '')
kiem('Mốc không cảnh báo = 3170', khong_canh_bao == 3170, '5 × số đơn trễ của validation.')
kiem('Đổi chi phí thành 2·FP + 5·FN: không cảnh báo rẻ hơn mọi ngưỡng', chon_tau(p_va, yva, 2, 5)[1].chi_phi.min() > khong_canh_bao, '')

## Bước 5 · Khoá τ, chấm test đúng một lần

Lưu ý trung thực: test này đã được xem ở bản thử nghiệm và ở v2 (test lịch sử). Báo cáo như minh hoạ việc áp quy tắc đã khoá, không phải đánh giá độc lập mới.

In [ ]:
_SO_LAN_CHAM = 0
def cham_test(tau):
    # Chấm test một lần cho τ ĐÃ KHOÁ trên validation
    global _SO_LAN_CHAM
    if _SO_LAN_CHAM >= 1:
        raise RuntimeError('Test đã chấm một lần. Muốn thử thêm: quay lại validation và ghi rõ trong báo cáo.')
    _SO_LAN_CHAM += 1
    return ma_tran(yte, sigmoid(b_gd + Xte @ w_gd) >= tau)

TAU_KHOA = TAU
KQ = cham_test(TAU_KHOA)
tn, fp, fn, tp = KQ
print('Test · τ =', TAU_KHOA, '· TN, FP, FN, TP =', KQ, '· precision', round(tp / (tp + fp), 4), '· recall', round(tp / (tp + fn), 4))
print('chi phí', fp + 5 * fn, 'so với không cảnh báo', 5 * yte.sum())

In [ ]:
# TỰ KIỂM · bước 5
kiem('cham_test được gọi đúng một lần', _SO_LAN_CHAM == 1, 'Restart & Run All nếu đã gọi nhiều lần.')
kiem('Test: (TN, FP, FN, TP) = (36429, 359, 3317, 161)', KQ == (36429, 359, 3317, 161), '')
kiem('Chi phí test thấp hơn không cảnh báo (17390)', fp + 5 * fn < 17390, '')

## Báo cáo một trang (nộp cùng notebook)

Mỗi mục 2–4 câu.

1. **Dữ liệu và nhãn.** Ba tập, số đơn, tỷ lệ trễ; nhãn theo ngày lịch; thời điểm dự đoán.
2. **Thuật toán.** Hàm mục tiêu (BCE + L2, λ = 1/(nC)); η, số vòng; bằng chứng hội tụ; mức khớp scikit-learn.
3. **Ba lớp đánh giá trên test.** Xếp hạng (AUC hoặc AP, kèm mốc), quyết định (bốn ô, chi phí so với không cảnh báo), xác suất (BCE so với dự đoán hằng số).
4. **Một đơn minh hoạ.** Chọn theo quy tắc khai báo trước; đóng góp từng đặc trưng vào z.
5. **Giới hạn.** Ít nhất ba: quy tắc “biết nhãn” kéo thấp tỷ lệ trễ train; tỷ lệ trễ đổi theo tháng; test lịch sử; chi phí giả định; không nhân quả.

## Nâng cao (tuỳ chọn)

- **Hiệu chuẩn:** chia test thành 10 nhóm theo p; vẽ p trung bình và tỷ lệ trễ thật; tỷ số giữa hai bên.
- **τ* = c_FP/(c_FP + c_FN) = 1/6:** so số cảnh báo và chi phí trên validation với τ đã chọn; giải thích vì sao khác.
- **Thiên lệch do quy tắc biết nhãn:** từ `olist_orders_dataset.csv` (Kaggle, hoặc data/olist.zip của Bài 5 v2), tính tỷ lệ trễ của mọi đơn đã giao duyệt tháng 10/2017 và của phần giao trước 01/11/2017.

In [ ]:
# Nâng cao · khởi đầu cho phần hiệu chuẩn (tuỳ chọn)
p_te = sigmoid(b_gd + Xte @ w_gd)
nhom = pd.qcut(p_te, 10, labels=False)
hc = pd.DataFrame({'p': p_te, 'y': yte, 'nhom': nhom}).groupby('nhom').agg(p_tb=('p', 'mean'), ty_le_tre=('y', 'mean'))
print(hc.round(4))